
# 05 — Multimodal Fusion

## Multimodal Medicinal Plant Leaf Retrieval Using Visual and Textual Features

---

## 1. Notebook Overview

This notebook focuses on integrating visual and textual features
for a multimodal medicinal plant leaf retrieval system.

The visual modality is represented using HSV Histogram,
Gray-Level Co-occurrence Matrix (GLCM), and Hu Moments.

The textual modality is represented using Term Frequency-Inverse
Document Frequency (TF-IDF) with two configurations:

- Unigram
- Unigram + Bigram

Visual feature matching is performed using Euclidean Distance,
while textual feature matching is performed using Cosine Similarity.

Both modality scores are integrated through a weighted
score-level late fusion approach to generate ranked retrieval results.

---

## 2. Objectives

The objectives of this notebook are:

1. Load previously extracted visual and textual feature artifacts.
2. Validate multimodal sample identities and feature alignment.
3. Calculate visual Euclidean distance matrices.
4. Analyze visual distance distributions.
5. Convert visual distances into similarity scores.
6. Calculate textual cosine similarity matrices.
7. Analyze cross-modality similarity score distributions.
8. Implement weighted score-level late fusion.
9. Analyze the sensitivity of different fusion weights.
10. Generate Top-K multimodal retrieval results.
11. Visualize image-only, text-only, and multimodal retrieval results.
12. Validate the mathematical and technical consistency of fusion.
13. Save multimodal fusion artifacts for subsequent evaluation.

---

## 3. Dataset Information

The multimodal dataset contains 104 paired image-text samples
representing five medicinal plant classes.

| Plant Class | Number of Samples |
|---|---:|
| Mangkok | 20 |
| Salam | 19 |
| Sambang Darah | 20 |
| Sirih | 20 |
| Tapak Kuda | 25 |
| **Total** | **104** |

Each sample contains:

- Unique sample ID
- Processed leaf image
- Binary segmentation mask
- Leaf description
- Plant class label
- Visual feature vector
- Textual feature vector

All modality features must follow the same sample ID order.

---

## 4. Feature Representation

### 4.1 Visual Features

| Feature Extraction Method | Feature Dimension |
|---|---:|
| HSV Histogram | 64 |
| GLCM | 72 |
| Hu Moments | 7 |
| **Total** | **143** |

Visual feature matrix:

$$
X_{image}\in\mathbb{R}^{104\times143}
$$

Visual features have been normalized and combined
through feature-level concatenation.

### 4.2 Textual Features

| TF-IDF Configuration | Feature Dimension |
|---|---:|
| Unigram | 68 |
| Unigram + Bigram | 261 |

Unigram feature matrix:

$$
X_{text}^{uni}\in\mathbb{R}^{104\times68}
$$

Unigram and bigram feature matrix:

$$
X_{text}^{uni+bi}\in\mathbb{R}^{104\times261}
$$

Both configurations are retained for comparison.

---

## 5. Visual Feature Matching

Visual feature matching is performed using Euclidean Distance.

$$
d(q,i)=
\sqrt{
\sum_{j=1}^{143}(q_j-i_j)^2
}
$$

Where:

- $q$ represents the visual query feature vector.
- $i$ represents the reference visual feature vector.
- $d(q,i)$ represents the Euclidean distance.

A smaller distance indicates greater similarity
in the visual feature space.

---

## 6. Visual Distance-to-Similarity Conversion

Euclidean Distance and Cosine Similarity have different
score interpretations.

Therefore, visual distances must be converted into
similarity scores before multimodal fusion.

An initial inverse-distance similarity function is defined as:

$$
S_{image}(q,i)=
\frac{1}{1+d(q,i)}
$$

The resulting similarity score satisfies:

$$
0<S_{image}\leq1
$$

A higher similarity score indicates a more similar
visual representation.

The suitability of the score conversion method will be
examined through score distribution analysis.

---

## 7. Text Feature Matching

Text feature matching is performed using Cosine Similarity.

$$
S_{text}(q,i)=
\frac{q\cdot i}
{\|q\|_2\|i\|_2}
$$

For nonnegative TF-IDF feature vectors:

$$
0\leq S_{text}\leq1
$$

Higher values indicate greater textual similarity.

Empty and all-OOV query vectors must not generate
arbitrary text-only retrieval rankings.

---

## 8. Multimodal Fusion Method

This project applies score-level late fusion.

Visual and textual features are processed independently
before their similarity scores are combined.

The final multimodal similarity score is calculated using:

$$
S_{final}(q,i)=
\alpha S_{image}(q,i)
+(1-\alpha)S_{text}(q,i)
$$

Where:

- $S_{image}$ is the visual similarity score.
- $S_{text}$ is the textual similarity score.
- $\alpha$ is the visual modality weight.
- $(1-\alpha)$ is the textual modality weight.
- $S_{final}$ is the final multimodal similarity score.

The weight constraint is:

$$
0\leq\alpha\leq1
$$

The main fusion scenarios are:

| Scenario | Visual Weight | Text Weight |
|---|---:|---:|
| Image Only | 1.0 | 0.0 |
| Text Only | 0.0 | 1.0 |
| Balanced Multimodal Fusion | 0.5 | 0.5 |

Additional weight configurations will be examined
through fusion weight sensitivity analysis.

---

## 9. Multimodal Retrieval Pipeline

```text
Multimodal Query
        |
        +-------------------------+
        |                         |
    Image Query               Text Query
        |                         |
 Image Preprocessing       Text Preprocessing
        |                         |
 HSV + GLCM + Hu Moments        TF-IDF
        |                         |
 Visual Normalization      Query Transformation
        |                         |
 Euclidean Distance       Cosine Similarity
        |                         |
 Distance-to-Similarity     Text Similarity
        |                         |
        +-----------+-------------+
                    |
        Weighted Score-Level Fusion
                    |
          Final Similarity Score
                    |
             Top-K Ranking
                    |
           Retrieval Results
```

---

## 10. Experimental Configurations

The following retrieval configurations are investigated:

| ID | Configuration |
|---|---|
| C01 | Image Only |
| C02 | Text Only — Unigram |
| C03 | Text Only — Unigram + Bigram |
| C04 | Multimodal — Image + Unigram |
| C05 | Multimodal — Image + Unigram + Bigram |

Fusion weights will be investigated using:

$$
\alpha\in
\{0.0,0.1,0.2,0.3,0.4,0.5,
0.6,0.7,0.8,0.9,1.0\}
$$

Top-K retrieval configurations:

$$
K\in\{1,3,5,10\}
$$

Image-only configurations shared across both
text representations will not be counted as
different retrieval methods.

---

## 11. Notebook Workflow

| Cell | Processing Stage |
|---:|---|
| 1 | Notebook Overview |
| 2 | Import Libraries |
| 3 | Path Configuration |
| 4 | Load Visual Feature Artifacts |
| 5 | Load Text Feature Artifacts |
| 6 | Multimodal Dataset Validation |
| 7 | Fusion Configuration |
| 8 | Euclidean Distance Calculation |
| 9 | Visual Distance Analysis |
| 10 | Visual Distance-to-Similarity Conversion |
| 11 | Text Cosine Similarity Calculation |
| 12 | Cross-Modality Score Distribution Analysis |
| 13 | Weighted Score-Level Late Fusion |
| 14 | Fusion Weight Sensitivity Analysis |
| 15 | Multimodal Similarity Matrix Analysis |
| 16 | Top-K Multimodal Retrieval |
| 17 | Retrieval Result Visualization |
| 18 | Multimodal Fusion Validation |
| 19 | Save Multimodal Fusion Artifacts |
| 20 | Multimodal Fusion Summary |

---

## 12. Expected Outputs

The expected outputs of this notebook include:

- Visual Euclidean distance matrices
- Visual similarity matrices
- Textual cosine similarity matrices
- Multimodal similarity matrices
- Cross-modality score distribution reports
- Fusion weight sensitivity reports
- Top-K retrieval results
- Comparative retrieval visualizations
- Mathematical validation reports
- Multimodal fusion configuration
- Saved fusion artifacts
- Final multimodal fusion summary

---

## 13. Methodological Considerations

The following methodological principles are maintained:

1. Visual and textual samples must be aligned by sample ID.
2. Visual and textual feature spaces remain separate.
3. All fusion scores must have consistent similarity direction.
4. Fusion weights must be nonnegative and sum to one.
5. Self-matches must be excluded when reference samples
   are used as exploratory queries.
6. Similarity ties must be handled consistently.
7. Empty or unusable modalities require an appropriate
   single-modality fallback or query rejection.
8. Original dataset review flags must be preserved.
9. Full-gallery analyses are exploratory and must not be
   reported as independent retrieval evaluation.
10. Final configuration and weight selection must use
    validation data rather than final test data.
11. TF-IDF vectorizers and visual normalizers must be
    refitted using reference data only during
    independent evaluation.
12. Duplicate descriptions and related specimens must
    be considered when constructing evaluation splits.

---

## 14. Research Hypothesis

Combining complementary visual and textual information
through score-level late fusion may improve medicinal
plant leaf retrieval compared with using either
modality independently.

This hypothesis will be investigated through comparative
retrieval experiments and subsequently evaluated using
Precision@K, Recall@K, and mean Average Precision (mAP).

---

## 15. Notebook Status

**Current Stage:** Multimodal Fusion

**Fusion Method:** Score-Level Late Fusion

**Fusion Strategy:** Weighted Sum

**Previous Notebook:** 04_text_feature_extraction.ipynb

**Next Notebook:** 06_retrieval_evaluation.ipynb

**Status:** In Progress

---


In [1]:
# ==================================================
# CELL 2 - IMPORT LIBRARIES
# ==================================================

# ==================================================
# 1. STANDARD LIBRARIES
# ==================================================

import os
import sys
import json
import time
import hashlib
import warnings
import platform
import itertools

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from importlib.metadata import version


# ==================================================
# 2. NUMERICAL COMPUTATION AND DATA PROCESSING
# ==================================================

import numpy as np
import pandas as pd

import scipy
from scipy.sparse import (
    csr_matrix,
    issparse,
    load_npz,
    save_npz
)


# ==================================================
# 3. DISTANCE AND SIMILARITY CALCULATION
# ==================================================

from scipy.spatial.distance import (
    cdist,
    pdist,
    squareform
)

from scipy.stats import (
    pearsonr,
    spearmanr
)

import sklearn

from sklearn.metrics.pairwise import (
    euclidean_distances,
    cosine_similarity
)

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler
)


# ==================================================
# 4. MODEL AND ARTIFACT MANAGEMENT
# ==================================================

import joblib


# ==================================================
# 5. IMAGE PROCESSING
# ==================================================

import cv2

from PIL import Image


# ==================================================
# 6. DATA VISUALIZATION
# ==================================================

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter


# ==================================================
# 7. NOTEBOOK UTILITIES
# ==================================================

from tqdm.auto import tqdm
from IPython.display import display, Markdown


# ==================================================
# 8. REPRODUCIBILITY CONFIGURATION
# ==================================================

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

EPSILON = 1e-10

FEATURE_DTYPE = np.float64

np.set_printoptions(
    precision=6,
    suppress=True
)

pd.set_option(
    "display.max_columns",
    50
)

pd.set_option(
    "display.max_rows",
    100
)

pd.set_option(
    "display.max_colwidth",
    150
)

pd.set_option(
    "display.precision",
    6
)


# ==================================================
# 9. VISUALIZATION CONFIGURATION
# ==================================================

sns.set_theme(
    style="whitegrid",
    context="notebook"
)

plt.rcParams.update({

    "figure.figsize": (12, 6),

    "figure.dpi": 120,

    "savefig.dpi": 300,

    "font.family": "DejaVu Sans",

    "font.size": 11,

    "axes.titlesize": 13,

    "axes.titleweight": "bold",

    "axes.labelsize": 11,

    "axes.spines.top": False,

    "axes.spines.right": False,

    "xtick.labelsize": 10,

    "ytick.labelsize": 10,

    "legend.fontsize": 10,

    "grid.alpha": 0.25,

    "figure.facecolor": "white",

    "axes.facecolor": "white",

    "savefig.facecolor": "white",

    "savefig.bbox": "tight"
})


# ==================================================
# 10. COLOR PALETTE CONFIGURATION
# ==================================================

CLASS_COLORS = {

    "Mangkok": "#66A58A",

    "Salam": "#83A7C7",

    "Sambang Darah": "#CBA461",

    "Sirih": "#998AB8",

    "Tapak Kuda": "#D97878"
}


MODALITY_COLORS = {

    "Image Only": "#66A58A",

    "Text Only": "#83A7C7",

    "Multimodal": "#CBA461"
}


TFIDF_COLORS = {

    "Unigram": "#66A58A",

    "Unigram + Bigram": "#83A7C7"
}


FEATURE_GROUP_COLORS = {

    "HSV": "#66A58A",

    "GLCM": "#83A7C7",

    "Hu Moments": "#CBA461"
}


RETRIEVAL_COLORS = {

    "Image Only": "#66A58A",

    "Text Only - Unigram": "#83A7C7",

    "Text Only - Unigram + Bigram": "#998AB8",

    "Multimodal - Unigram": "#CBA461",

    "Multimodal - Unigram + Bigram": "#D97878"
}


VALIDATION_COLORS = {

    "Passed": "#66A58A",

    "Pending Review": "#CBA461",

    "Failed": "#D97878"
}


PAIR_TYPE_COLORS = {

    "Within Class": "#66A58A",

    "Cross Class": "#83A7C7"
}


SIMILARITY_CMAP = "viridis"

DISTANCE_CMAP = "magma_r"

DIVERGING_CMAP = "RdBu_r"


# ==================================================
# 11. SOFTWARE ENVIRONMENT INFORMATION
# ==================================================

library_versions = {

    "Python": platform.python_version(),

    "NumPy": np.__version__,

    "Pandas": pd.__version__,

    "SciPy": scipy.__version__,

    "Scikit-learn": sklearn.__version__,

    "Joblib": joblib.__version__,

    "OpenCV": cv2.__version__,

    "Pillow": version("Pillow"),

    "Matplotlib": matplotlib.__version__,

    "Seaborn": sns.__version__,

    "TQDM": version("tqdm"),

    "IPython": version("ipython")
}


library_versions_df = pd.DataFrame(

    library_versions.items(),

    columns=[
        "Library",
        "Version"
    ]
)


# ==================================================
# 12. MATHEMATICAL FUNCTION SANITY CHECK
# ==================================================

# Synthetic vectors for basic mathematical validation.
# These are not actual dataset features.

test_vector_a = np.array([
    [1.0, 0.0, 0.0]
])

test_vector_b = np.array([
    [0.0, 1.0, 0.0]
])

test_vector_c = np.array([
    [1.0, 0.0, 0.0]
])


# A. Euclidean Distance validation.

test_euclidean_same = float(

    euclidean_distances(
        test_vector_a,
        test_vector_c
    )[0, 0]

)

test_euclidean_different = float(

    euclidean_distances(
        test_vector_a,
        test_vector_b
    )[0, 0]

)


# B. Cosine Similarity validation.

test_cosine_same = float(

    cosine_similarity(
        test_vector_a,
        test_vector_c
    )[0, 0]

)

test_cosine_orthogonal = float(

    cosine_similarity(
        test_vector_a,
        test_vector_b
    )[0, 0]

)


# C. Distance-to-similarity validation.

test_visual_similarity_same = (

    1.0 / (1.0 + test_euclidean_same)

)

test_visual_similarity_different = (

    1.0 / (1.0 + test_euclidean_different)

)


# D. Weighted score fusion validation.

test_image_score = 0.8
test_text_score = 0.6
test_alpha = 0.5

test_fusion_score = (

    test_alpha * test_image_score

    +

    (1 - test_alpha) * test_text_score

)


# ==================================================
# 13. SPARSE MATRIX SANITY CHECK
# ==================================================

test_dense_matrix = np.array([

    [1.0, 0.0, 0.0],

    [0.0, 1.0, 0.0]

])

test_sparse_matrix = csr_matrix(
    test_dense_matrix
)

test_sparse_similarity = cosine_similarity(
    test_sparse_matrix
)

test_dense_similarity = cosine_similarity(
    test_dense_matrix
)


# ==================================================
# 14. REGISTER LIBRARY VALIDATION RESULTS
# ==================================================

fusion_library_checks = [

    {
        "validation_check":
            "Euclidean distance of identical vectors",

        "expected": 0.0,

        "actual": test_euclidean_same,

        "passed": np.isclose(
            test_euclidean_same,
            0.0,
            atol=EPSILON
        )
    },

    {
        "validation_check":
            "Euclidean distance of orthogonal unit vectors",

        "expected": np.sqrt(2),

        "actual": test_euclidean_different,

        "passed": np.isclose(
            test_euclidean_different,
            np.sqrt(2),
            atol=EPSILON
        )
    },

    {
        "validation_check":
            "Cosine similarity of identical vectors",

        "expected": 1.0,

        "actual": test_cosine_same,

        "passed": np.isclose(
            test_cosine_same,
            1.0,
            atol=EPSILON
        )
    },

    {
        "validation_check":
            "Cosine similarity of orthogonal vectors",

        "expected": 0.0,

        "actual": test_cosine_orthogonal,

        "passed": np.isclose(
            test_cosine_orthogonal,
            0.0,
            atol=EPSILON
        )
    },

    {
        "validation_check":
            "Zero distance produces similarity one",

        "expected": 1.0,

        "actual": test_visual_similarity_same,

        "passed": np.isclose(
            test_visual_similarity_same,
            1.0,
            atol=EPSILON
        )
    },

    {
        "validation_check":
            "Inverse-distance similarity calculation",

        "expected": 1 / (1 + np.sqrt(2)),

        "actual": test_visual_similarity_different,

        "passed": np.isclose(

            test_visual_similarity_different,

            1 / (1 + np.sqrt(2)),

            atol=EPSILON
        )
    },

    {
        "validation_check":
            "Weighted score fusion calculation",

        "expected": 0.7,

        "actual": test_fusion_score,

        "passed": np.isclose(
            test_fusion_score,
            0.7,
            atol=EPSILON
        )
    },

    {
        "validation_check":
            "Sparse and dense cosine similarity consistency",

        "expected": 0.0,

        "actual": float(

            np.max(

                np.abs(

                    test_sparse_similarity

                    - test_dense_similarity

                )

            )
        ),

        "passed": np.allclose(

            test_sparse_similarity,

            test_dense_similarity,

            atol=EPSILON,

            rtol=0
        )
    }
]


fusion_library_validation_df = pd.DataFrame(
    fusion_library_checks
)

fusion_library_validation_df["status"] = np.where(

    fusion_library_validation_df["passed"],

    "Passed",

    "Failed"
)


# ==================================================
# 15. VALIDATE LIBRARY ENVIRONMENT
# ==================================================

FUSION_LIBRARIES_IMPORTED = False

assert fusion_library_validation_df[
    "passed"
].all(), (

    fusion_library_validation_df.loc[

        ~fusion_library_validation_df["passed"]

    ].to_string(index=False)
)

FUSION_LIBRARIES_IMPORTED = True


# ==================================================
# 16. PRINT IMPORT SUMMARY
# ==================================================

print("=" * 70)

print("MULTIMODAL FUSION - LIBRARY INITIALIZATION")

print("=" * 70)


print("\nSYSTEM INFORMATION")

print("-" * 70)

print(f"Operating system : {platform.system()}")

print(f"OS release       : {platform.release()}")

print(f"Python version   : {platform.python_version()}")

print(f"Architecture     : {platform.machine()}")

print(f"Random state     : {RANDOM_STATE}")

print(f"Feature dtype    : {FEATURE_DTYPE.__name__}")

print(f"Tolerance        : {EPSILON}")


print("\nLIBRARY VERSIONS")

print("-" * 70)

display(library_versions_df)


print("\nMATHEMATICAL FUNCTION VALIDATION")

print("-" * 70)

display(
    fusion_library_validation_df.round(8)
)


print("\nIMPORT STATUS")

print("-" * 70)

print("[PASS] Standard libraries imported.")

print("[PASS] Numerical computation libraries imported.")

print("[PASS] Sparse matrix utilities imported.")

print("[PASS] Euclidean distance functions imported.")

print("[PASS] Cosine similarity functions imported.")

print("[PASS] Statistical analysis libraries imported.")

print("[PASS] Model and artifact utilities imported.")

print("[PASS] Image processing libraries imported.")

print("[PASS] Visualization libraries imported.")

print("[PASS] Notebook utilities imported.")

print("[PASS] Reproducibility configuration initialized.")

print("[PASS] Visualization configuration initialized.")

print("[PASS] Mathematical sanity checks completed.")


print("\nVALIDATION SUMMARY")

print("-" * 70)

print(

    f"Total checks  : "

    f"{len(fusion_library_validation_df)}"

)

print(

    f"Passed checks : "

    f"{fusion_library_validation_df['passed'].sum()}"

)

print(

    f"Failed checks : "

    f"{(~fusion_library_validation_df['passed']).sum()}"

)


print("\n" + "=" * 70)

print("ALL LIBRARIES IMPORTED SUCCESSFULLY")

print("=" * 70)

print("\nNext cell: Cell 3 - Path Configuration")

MULTIMODAL FUSION - LIBRARY INITIALIZATION

SYSTEM INFORMATION
----------------------------------------------------------------------
Operating system : Windows
OS release       : 11
Python version   : 3.13.14
Architecture     : AMD64
Random state     : 42
Feature dtype    : float64
Tolerance        : 1e-10

LIBRARY VERSIONS
----------------------------------------------------------------------


,Library,Version
0,Python,3.13.14
1,NumPy,2.5.5
2,Pandas,3.1.1
3,SciPy,1.18.1
4,Scikit-learn,1.9.1
5,Joblib,1.6.0
6,OpenCV,5.1.0
7,Pillow,13.0.0
8,Matplotlib,3.11.2
9,Seaborn,0.13.2



MATHEMATICAL FUNCTION VALIDATION
----------------------------------------------------------------------


,validation_check,expected,actual,passed,status
0,Euclidean distance of identical vectors,0.000000,0.000000,True,Passed
1,Euclidean distance of orthogonal unit vectors,1.414214,1.414214,True,Passed
2,Cosine similarity of identical vectors,1.000000,1.000000,True,Passed
3,Cosine similarity of orthogonal vectors,0.000000,0.000000,True,Passed
4,Zero distance produces similarity one,1.000000,1.000000,True,Passed
5,Inverse-distance similarity calculation,0.414214,0.414214,True,Passed
6,Weighted score fusion calculation,0.700000,0.700000,True,Passed
7,Sparse and dense cosine similarity consistency,0.000000,0.000000,True,Passed



IMPORT STATUS
----------------------------------------------------------------------
[PASS] Standard libraries imported.
[PASS] Numerical computation libraries imported.
[PASS] Sparse matrix utilities imported.
[PASS] Euclidean distance functions imported.
[PASS] Cosine similarity functions imported.
[PASS] Statistical analysis libraries imported.
[PASS] Model and artifact utilities imported.
[PASS] Image processing libraries imported.
[PASS] Visualization libraries imported.
[PASS] Notebook utilities imported.
[PASS] Reproducibility configuration initialized.
[PASS] Visualization configuration initialized.
[PASS] Mathematical sanity checks completed.

VALIDATION SUMMARY
----------------------------------------------------------------------
Total checks  : 8
Passed checks : 8
Failed checks : 0

ALL LIBRARIES IMPORTED SUCCESSFULLY

Next cell: Cell 3 - Path Configuration


In [2]:
# ==================================================
# CELL 3 - PATH CONFIGURATION
# ==================================================

from tempfile import NamedTemporaryFile

print("MULTIMODAL FUSION - PATH CONFIGURATION")
print("=" * 70)

FUSION_PATH_CONFIGURATION_COMPLETED = False


# ==================================================
# 1. VALIDATE PREVIOUS CELL
# ==================================================

assert FUSION_LIBRARIES_IMPORTED is True, (
    "Run Cell 2 - Import Libraries first."
)

assert fusion_library_validation_df["passed"].all()

print("[PASS] Library initialization validated.")


# ==================================================
# 2. AUTOMATIC PROJECT ROOT DETECTION
# ==================================================

# Set a manual path only if automatic detection fails.
# Example:
# MANUAL_PROJECT_ROOT = r"D:\Project\Your_Project"

MANUAL_PROJECT_ROOT = None


def detect_project_root(start_path=None):

    start_path = Path(
        start_path or Path.cwd()
    ).resolve()

    candidates = [
        start_path,
        *start_path.parents
    ]

    for candidate in candidates:

        metadata_file = (
            candidate
            / "dataset"
            / "processed"
            / "metadata_processed.csv"
        )

        if metadata_file.is_file():

            return candidate

    raise FileNotFoundError(
        "Project root could not be detected. "
        "Set MANUAL_PROJECT_ROOT to the project directory."
    )


if MANUAL_PROJECT_ROOT is not None:

    PROJECT_ROOT = Path(
        MANUAL_PROJECT_ROOT
    ).expanduser().resolve()

else:

    PROJECT_ROOT = detect_project_root()


assert PROJECT_ROOT.is_dir(), (
    f"Invalid project root: {PROJECT_ROOT}"
)

print("\n1. PROJECT ROOT DETECTION")
print("-" * 70)

print(f"Working directory : {Path.cwd()}")
print(f"Project root      : {PROJECT_ROOT}")

print("[PASS] Project root detected successfully.")


# ==================================================
# 3. MAIN PROJECT DIRECTORIES
# ==================================================

DATASET_DIR = PROJECT_ROOT / "dataset"

PROCESSED_DIR = DATASET_DIR / "processed"

FEATURES_DIR = DATASET_DIR / "features"

VISUAL_FEATURES_DIR = FEATURES_DIR / "visual"

TEXT_FEATURES_DIR = FEATURES_DIR / "text"

FUSION_FEATURES_DIR = FEATURES_DIR / "fusion"

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

RESULTS_DIR = PROJECT_ROOT / "results"

FIGURES_DIR = RESULTS_DIR / "figures"

METRICS_DIR = RESULTS_DIR / "metrics"

FUSION_FIGURES_DIR = (
    FIGURES_DIR / "multimodal_fusion"
)

# Alias for consistency with previous notebooks.

MULTIMODAL_FUSION_DIR = FUSION_FEATURES_DIR

MULTIMODAL_FUSION_FIGURES_DIR = FUSION_FIGURES_DIR


# ==================================================
# 4. CONFIGURE METADATA PATHS
# ==================================================

PROCESSED_METADATA_PATH = (
    PROCESSED_DIR
    / "metadata_processed.csv"
)

VISUAL_METADATA_PATH = (
    VISUAL_FEATURES_DIR
    / "visual_feature_metadata.csv"
)

MULTIMODAL_METADATA_PATH = (
    TEXT_FEATURES_DIR
    / "multimodal_feature_metadata.csv"
)


# ==================================================
# 5. CONFIGURE VISUAL FEATURE ARTIFACTS
# ==================================================

VISUAL_ARTIFACT_PATHS = {

    "hsv_features":
        VISUAL_FEATURES_DIR
        / "hsv_features.npy",

    "glcm_features":
        VISUAL_FEATURES_DIR
        / "glcm_features.npy",

    "hu_features":
        VISUAL_FEATURES_DIR
        / "hu_features.npy",

    "hu_raw_moments":
        VISUAL_FEATURES_DIR
        / "hu_raw_moments.npy",

    "raw_features":
        VISUAL_FEATURES_DIR
        / "visual_features_raw.npy",

    "baseline_features":
        VISUAL_FEATURES_DIR
        / "visual_features_baseline.npy",

    "normalized_features":
        VISUAL_FEATURES_DIR
        / "visual_features_normalized.npy",

    "sample_ids":
        VISUAL_FEATURES_DIR
        / "visual_sample_ids.npy",

    "metadata":
        VISUAL_METADATA_PATH,

    "feature_index":
        VISUAL_FEATURES_DIR
        / "visual_feature_index.csv",

    "normalizer":
        VISUAL_FEATURES_DIR
        / "visual_normalizer.joblib",

    "baseline_normalizer":
        VISUAL_FEATURES_DIR
        / "baseline_visual_normalizer.joblib",

    "feature_names":
        VISUAL_FEATURES_DIR
        / "visual_feature_names.json",

    "configuration":
        VISUAL_FEATURES_DIR
        / "visual_feature_config.json"
}


# Main visual feature aliases.

VISUAL_RAW_FEATURES_PATH = (
    VISUAL_ARTIFACT_PATHS["raw_features"]
)

VISUAL_BASELINE_FEATURES_PATH = (
    VISUAL_ARTIFACT_PATHS["baseline_features"]
)

VISUAL_NORMALIZED_FEATURES_PATH = (
    VISUAL_ARTIFACT_PATHS["normalized_features"]
)

VISUAL_SAMPLE_IDS_PATH = (
    VISUAL_ARTIFACT_PATHS["sample_ids"]
)

VISUAL_NORMALIZER_PATH = (
    VISUAL_ARTIFACT_PATHS["normalizer"]
)

VISUAL_CONFIG_PATH = (
    VISUAL_ARTIFACT_PATHS["configuration"]
)


# ==================================================
# 6. CONFIGURE TF-IDF ARTIFACTS
# ==================================================

TEXT_CONFIGURATION_SLUGS = {

    "Unigram": "unigram",

    "Unigram + Bigram": "unigram_bigram"
}

TEXT_ARTIFACT_PATHS = {}

for configuration_name, slug in (
    TEXT_CONFIGURATION_SLUGS.items()
):

    TEXT_ARTIFACT_PATHS[
        configuration_name
    ] = {

        "matrix":
            TEXT_FEATURES_DIR
            / f"tfidf_{slug}_features.npz",

        "vectorizer":
            TEXT_FEATURES_DIR
            / f"tfidf_{slug}_vectorizer.joblib",

        "feature_names":
            TEXT_FEATURES_DIR
            / f"tfidf_{slug}_feature_names.npy",

        "idf_weights":
            TEXT_FEATURES_DIR
            / f"tfidf_{slug}_idf_weights.npy",

        "vocabulary":
            TEXT_FEATURES_DIR
            / f"tfidf_{slug}_vocabulary.csv"
    }


# ==================================================
# 7. CONFIGURE SHARED TEXT ARTIFACTS
# ==================================================

TEXT_SHARED_ARTIFACT_PATHS = {

    "sample_ids":
        TEXT_FEATURES_DIR
        / "text_sample_ids.npy",

    "class_labels":
        TEXT_FEATURES_DIR
        / "text_class_labels.npy",

    "metadata":
        MULTIMODAL_METADATA_PATH,

    "corpus":
        TEXT_FEATURES_DIR
        / "text_corpus.json",

    "preprocessing_config":
        TEXT_FEATURES_DIR
        / "text_query_preprocessing_config.json",

    "feature_config":
        TEXT_FEATURES_DIR
        / "text_feature_extraction_config.json",

    "manifest":
        TEXT_FEATURES_DIR
        / "text_feature_artifact_manifest.csv"
}


# Main text feature aliases.

TFIDF_UNIGRAM_FEATURES_PATH = (
    TEXT_ARTIFACT_PATHS["Unigram"]["matrix"]
)

TFIDF_BIGRAM_FEATURES_PATH = (
    TEXT_ARTIFACT_PATHS["Unigram + Bigram"]["matrix"]
)

TFIDF_UNIGRAM_VECTORIZER_PATH = (
    TEXT_ARTIFACT_PATHS["Unigram"]["vectorizer"]
)

TFIDF_BIGRAM_VECTORIZER_PATH = (
    TEXT_ARTIFACT_PATHS["Unigram + Bigram"]["vectorizer"]
)

TEXT_SAMPLE_IDS_PATH = (
    TEXT_SHARED_ARTIFACT_PATHS["sample_ids"]
)

TEXT_CLASS_LABELS_PATH = (
    TEXT_SHARED_ARTIFACT_PATHS["class_labels"]
)

TEXT_CORPUS_PATH = (
    TEXT_SHARED_ARTIFACT_PATHS["corpus"]
)

TEXT_CONFIG_PATH = (
    TEXT_SHARED_ARTIFACT_PATHS["feature_config"]
)

TEXT_PREPROCESSING_CONFIG_PATH = (
    TEXT_SHARED_ARTIFACT_PATHS["preprocessing_config"]
)

TEXT_MANIFEST_PATH = (
    TEXT_SHARED_ARTIFACT_PATHS["manifest"]
)


# ==================================================
# 8. CONFIGURE MANIFEST AND HANDOFF PATHS
# ==================================================

VISUAL_MANIFEST_JSON_PATH = (
    VISUAL_FEATURES_DIR
    / "visual_feature_manifest.json"
)

VISUAL_MANIFEST_CSV_PATH = (
    METRICS_DIR
    / "174_visual_feature_artifact_manifest.csv"
)

TEXT_HANDOFF_PATH = (
    METRICS_DIR
    / "347_text_feature_extraction_handoff.json"
)

FUSION_SUPPORTING_PATHS = {

    "Visual Manifest JSON":
        VISUAL_MANIFEST_JSON_PATH,

    "Visual Manifest CSV":
        VISUAL_MANIFEST_CSV_PATH,

    "Text Notebook Handoff":
        TEXT_HANDOFF_PATH
}


# ==================================================
# 9. REGISTER REQUIRED INPUT FILES
# ==================================================

fusion_required_inputs = [

    (
        "Processed Metadata",
        "Dataset",
        PROCESSED_METADATA_PATH
    )
]

# Visual payload artifacts.

for name, path in VISUAL_ARTIFACT_PATHS.items():

    fusion_required_inputs.append((

        name,

        "Visual",

        path
    ))


# Configuration-specific text artifacts.

for configuration_name, paths in (
    TEXT_ARTIFACT_PATHS.items()
):

    for name, path in paths.items():

        fusion_required_inputs.append((

            name,

            f"Text - {configuration_name}",

            path
        ))


# Shared text artifacts and manifest.

for name, path in (
    TEXT_SHARED_ARTIFACT_PATHS.items()
):

    fusion_required_inputs.append((

        name,

        "Text - Shared",

        path
    ))


# ==================================================
# 10. VALIDATE REQUIRED INPUT DIRECTORIES
# ==================================================

fusion_input_directories = {

    "Dataset":
        DATASET_DIR,

    "Processed Dataset":
        PROCESSED_DIR,

    "Feature Dataset":
        FEATURES_DIR,

    "Visual Features":
        VISUAL_FEATURES_DIR,

    "Text Features":
        TEXT_FEATURES_DIR
}

fusion_input_directory_records = []

for name, path in (
    fusion_input_directories.items()
):

    available = path.is_dir()

    fusion_input_directory_records.append({

        "directory": name,

        "relative_path": (
            path.relative_to(PROJECT_ROOT).as_posix()
        ),

        "exists": available,

        "passed": available,

        "status": (
            "Available"
            if available
            else "Missing"
        )
    })


fusion_input_directory_df = pd.DataFrame(
    fusion_input_directory_records
)

print("\n2. INPUT DIRECTORY VALIDATION")
print("-" * 70)

display(fusion_input_directory_df)


# ==================================================
# 11. VALIDATE REQUIRED INPUT ARTIFACTS
# ==================================================

fusion_input_file_records = []

for name, modality, path in (
    fusion_required_inputs
):

    exists = path.is_file()

    file_size = (
        path.stat().st_size
        if exists
        else 0
    )

    valid = (
        exists
        and file_size > 0
    )

    fusion_input_file_records.append({

        "artifact": name,

        "modality": modality,

        "filename": path.name,

        "relative_path": (
            path.relative_to(PROJECT_ROOT).as_posix()
        ),

        "exists": exists,

        "size_bytes": file_size,

        "size_kib": file_size / 1024,

        "passed": valid,

        "status": (

            "Available"
            if valid
            else "Empty File"
            if exists
            else "Missing"
        )
    })


fusion_input_file_validation_df = pd.DataFrame(
    fusion_input_file_records
)

print("\n3. REQUIRED INPUT FILE VALIDATION")
print("-" * 70)

display(
    fusion_input_file_validation_df[
        [
            "modality",
            "filename",
            "size_kib",
            "status"
        ]
    ].round(4)
)


# ==================================================
# 12. VALIDATE SUPPORTING ARTIFACTS
# ==================================================

fusion_supporting_file_records = []

for name, path in (
    FUSION_SUPPORTING_PATHS.items()
):

    available = (
        path.is_file()
        and path.stat().st_size > 0
    )

    fusion_supporting_file_records.append({

        "artifact": name,

        "filename": path.name,

        "relative_path": (
            path.relative_to(PROJECT_ROOT).as_posix()
        ),

        "available": available,

        "status": (
            "Available"
            if available
            else "Unavailable"
        )
    })


fusion_supporting_file_df = pd.DataFrame(
    fusion_supporting_file_records
)

visual_manifest_available = any(

    path.is_file()
    and path.stat().st_size > 0

    for path in [
        VISUAL_MANIFEST_JSON_PATH,
        VISUAL_MANIFEST_CSV_PATH
    ]
)

print("\n4. SUPPORTING ARTIFACTS")
print("-" * 70)

display(fusion_supporting_file_df)

print(
    "Visual manifest available:",
    visual_manifest_available
)


# ==================================================
# 13. CREATE OUTPUT DIRECTORIES
# ==================================================

fusion_output_directories = {

    "Fusion Features":
        FUSION_FEATURES_DIR,

    "Fusion Figures":
        FUSION_FIGURES_DIR,

    "Metrics":
        METRICS_DIR
}

fusion_output_directory_records = []

for name, path in (
    fusion_output_directories.items()
):

    error_message = ""

    try:

        path.mkdir(
            parents=True,
            exist_ok=True
        )

        # Test actual write access using a temporary file.
        # Existing project artifacts are not modified.

        with NamedTemporaryFile(

            mode="w",

            prefix=".fusion_write_test_",

            suffix=".tmp",

            dir=path,

            encoding="utf-8",

            delete=True

        ) as temporary_file:

            temporary_file.write(
                "Multimodal fusion write test."
            )

            temporary_file.flush()

        writable = True

    except OSError as error:

        writable = False

        error_message = str(error)

    fusion_output_directory_records.append({

        "directory": name,

        "relative_path": (
            path.relative_to(PROJECT_ROOT).as_posix()
        ),

        "exists": path.is_dir(),

        "writable": writable,

        "passed": (
            path.is_dir()
            and writable
        ),

        "error": error_message,

        "status": (
            "Ready"
            if path.is_dir() and writable
            else "Failed"
        )
    })


fusion_output_directory_df = pd.DataFrame(
    fusion_output_directory_records
)

print("\n5. OUTPUT DIRECTORY VALIDATION")
print("-" * 70)

display(fusion_output_directory_df)


# ==================================================
# 14. SUMMARIZE INPUT ARTIFACT AVAILABILITY
# ==================================================

fusion_input_artifact_summary_df = (

    fusion_input_file_validation_df

    .groupby("modality", sort=False)

    .agg(

        required_files=(
            "filename",
            "count"
        ),

        available_files=(
            "passed",
            "sum"
        ),

        total_size_bytes=(
            "size_bytes",
            "sum"
        ),

        total_size_kib=(
            "size_kib",
            "sum"
        )
    )

    .reset_index()
)

fusion_input_artifact_summary_df[
    "missing_or_invalid_files"
] = (

    fusion_input_artifact_summary_df[
        "required_files"
    ]

    -

    fusion_input_artifact_summary_df[
        "available_files"
    ]
)

print("\n6. INPUT ARTIFACT SUMMARY")
print("-" * 70)

display(
    fusion_input_artifact_summary_df.round(4)
)


# ==================================================
# 15. CONFIGURE PLANNED FUSION OUTPUT ARTIFACTS
# ==================================================

# These paths will be used by subsequent cells.
# No fusion matrices are generated at this stage.

FUSION_OUTPUT_ARTIFACT_PATHS = {

    "visual_distance_matrix":
        FUSION_FEATURES_DIR
        / "visual_euclidean_distance_matrix.npy",

    "visual_similarity_matrix":
        FUSION_FEATURES_DIR
        / "visual_similarity_matrix.npy",

    "text_unigram_similarity_matrix":
        FUSION_FEATURES_DIR
        / "text_unigram_cosine_similarity_matrix.npy",

    "text_combined_similarity_matrix":
        FUSION_FEATURES_DIR
        / "text_combined_cosine_similarity_matrix.npy",

    "fusion_unigram_matrices":
        FUSION_FEATURES_DIR
        / "fusion_unigram_similarity_matrices.npz",

    "fusion_combined_matrices":
        FUSION_FEATURES_DIR
        / "fusion_combined_similarity_matrices.npz",

    "sample_ids":
        FUSION_FEATURES_DIR
        / "fusion_sample_ids.npy",

    "class_labels":
        FUSION_FEATURES_DIR
        / "fusion_class_labels.npy",

    "metadata":
        FUSION_FEATURES_DIR
        / "fusion_metadata.csv",

    "configuration":
        FUSION_FEATURES_DIR
        / "multimodal_fusion_config.json",

    "manifest":
        FUSION_FEATURES_DIR
        / "multimodal_fusion_artifact_manifest.csv"
}

fusion_planned_artifacts_df = pd.DataFrame([

    {

        "artifact": name,

        "filename": path.name,

        "relative_path": (
            path.relative_to(PROJECT_ROOT).as_posix()
        ),

        "already_exists": path.is_file()

    }

    for name, path in (
        FUSION_OUTPUT_ARTIFACT_PATHS.items()
    )
])

print("\n7. PLANNED FUSION ARTIFACTS")
print("-" * 70)

display(fusion_planned_artifacts_df)


# ==================================================
# 16. FINAL PATH CONFIGURATION VALIDATION
# ==================================================

fusion_path_checks = {

    "Project root detected":

        PROJECT_ROOT.is_dir(),

    "Required input directories available":

        fusion_input_directory_df["passed"].all(),

    "All required input artifacts available":

        fusion_input_file_validation_df["passed"].all(),

    "All required input files nonempty":

        fusion_input_file_validation_df[
            "size_bytes"
        ].gt(0).all(),

    "Visual artifact manifest available":

        visual_manifest_available,

    "Text artifact manifest available":

        TEXT_MANIFEST_PATH.is_file()
        and TEXT_MANIFEST_PATH.stat().st_size > 0,

    "Output directories available":

        fusion_output_directory_df["exists"].all(),

    "Output directories writable":

        fusion_output_directory_df["writable"].all(),

    "Required input paths unique":

        len({
            path.resolve()
            for _, _, path in fusion_required_inputs
        }) == len(fusion_required_inputs),

    "Planned output paths unique":

        len({
            path.resolve()
            for path in FUSION_OUTPUT_ARTIFACT_PATHS.values()
        }) == len(FUSION_OUTPUT_ARTIFACT_PATHS),

    "Input and output paths do not overlap":

        {
            path.resolve()
            for _, _, path in fusion_required_inputs
        }.isdisjoint({

            path.resolve()
            for path in FUSION_OUTPUT_ARTIFACT_PATHS.values()
        }),

    "Previous library checks passed":

        fusion_library_validation_df["passed"].all()
}

fusion_path_validation_df = pd.DataFrame([

    {

        "validation_check": check_name,

        "passed": bool(passed),

        "status": (
            "Passed" if passed else "Failed"
        )
    }

    for check_name, passed in (
        fusion_path_checks.items()
    )
])

fusion_path_total_checks = len(
    fusion_path_validation_df
)

fusion_path_passed_checks = int(

    fusion_path_validation_df["passed"].sum()
)

fusion_path_failed_checks = (

    fusion_path_total_checks

    - fusion_path_passed_checks
)

print("\n8. PATH CONFIGURATION VALIDATION")
print("-" * 70)

display(fusion_path_validation_df)


# ==================================================
# 17. CREATE PATH CONFIGURATION SUMMARY
# ==================================================

fusion_path_summary_df = pd.DataFrame([

    (
        "Project root detected",
        PROJECT_ROOT.is_dir()
    ),

    (
        "Required input directories",
        len(fusion_input_directory_df)
    ),

    (
        "Available input directories",
        int(
            fusion_input_directory_df["passed"].sum()
        )
    ),

    (
        "Required visual payload artifacts",
        len(VISUAL_ARTIFACT_PATHS)
    ),

    (
        "Required text artifacts including manifest",
        (
            sum(
                len(paths)
                for paths in TEXT_ARTIFACT_PATHS.values()
            )

            + len(TEXT_SHARED_ARTIFACT_PATHS)
        )
    ),

    (
        "Total required input files",
        len(fusion_input_file_validation_df)
    ),

    (
        "Available required input files",
        int(
            fusion_input_file_validation_df[
                "passed"
            ].sum()
        )
    ),

    (
        "Missing or invalid input files",
        int(
            (
                ~fusion_input_file_validation_df["passed"]
            ).sum()
        )
    ),

    (
        "Visual manifest available",
        visual_manifest_available
    ),

    (
        "Text handoff available",
        TEXT_HANDOFF_PATH.is_file()
        and TEXT_HANDOFF_PATH.stat().st_size > 0
    ),

    (
        "Supporting artifact paths",
        len(fusion_supporting_file_df)
    ),

    (
        "Available supporting artifacts",
        int(
            fusion_supporting_file_df["available"].sum()
        )
    ),

    (
        "Output directories",
        len(fusion_output_directory_df)
    ),

    (
        "Writable output directories",
        int(
            fusion_output_directory_df["writable"].sum()
        )
    ),

    (
        "Planned fusion artifacts",
        len(FUSION_OUTPUT_ARTIFACT_PATHS)
    ),

    (
        "Total validation checks",
        fusion_path_total_checks
    ),

    (
        "Validation checks passed",
        fusion_path_passed_checks
    ),

    (
        "Validation checks failed",
        fusion_path_failed_checks
    ),

    ("Visual features loaded", False),

    ("Text features loaded", False),

    ("Artifact checksums verified", False),

    ("Multimodal alignment validated", False),

    ("Fusion calculation completed", False),

    (
        "Path configuration checks passed",
        fusion_path_failed_checks == 0
    )

], columns=["Indicator", "Result"])

print("\n9. PATH CONFIGURATION SUMMARY")
print("-" * 70)

display(fusion_path_summary_df)


# ==================================================
# 18. SAVE PATH CONFIGURATION REPORTS
# ==================================================

fusion_path_reports = {

    "348_fusion_input_directory_validation.csv":
        fusion_input_directory_df,

    "349_fusion_input_file_validation.csv":
        fusion_input_file_validation_df,

    "350_fusion_supporting_artifact_validation.csv":
        fusion_supporting_file_df,

    "351_fusion_output_directory_validation.csv":
        fusion_output_directory_df,

    "352_fusion_input_artifact_summary.csv":
        fusion_input_artifact_summary_df,

    "353_fusion_planned_artifacts.csv":
        fusion_planned_artifacts_df,

    "354_fusion_path_validation.csv":
        fusion_path_validation_df,

    "355_fusion_path_configuration_summary.csv":
        fusion_path_summary_df,

    "356_fusion_library_versions.csv":
        library_versions_df,

    "357_fusion_library_validation.csv":
        fusion_library_validation_df
}

print("\n10. SAVING PATH CONFIGURATION REPORTS")
print("-" * 70)

metrics_writable = bool(

    fusion_output_directory_df.loc[

        fusion_output_directory_df[
            "directory"
        ].eq("Metrics"),

        "writable"

    ].iloc[0]
)

assert metrics_writable, (
    f"Cannot write validation reports to {METRICS_DIR}"
)

for filename, dataframe in (
    fusion_path_reports.items()
):

    output_path = METRICS_DIR / filename

    dataframe.to_csv(

        output_path,

        index=False,

        encoding="utf-8-sig",

        float_format="%.17g"
    )

    print(f"[SAVED] {filename}")


# ==================================================
# 19. FINAL COMPLETION CHECKS
# ==================================================

# Print invalid input files before stopping.

invalid_input_files = (

    fusion_input_file_validation_df.loc[

        ~fusion_input_file_validation_df["passed"]

    ]
)

if not invalid_input_files.empty:

    print("\n[ERROR] Missing or invalid input artifacts:")

    display(

        invalid_input_files[
            [
                "modality",
                "filename",
                "relative_path",
                "status"
            ]
        ]
    )

assert fusion_path_validation_df["passed"].all(), (

    fusion_path_validation_df.loc[

        ~fusion_path_validation_df["passed"]

    ].to_string(index=False)
)

assert all(

    (METRICS_DIR / filename).is_file()

    and (
        METRICS_DIR / filename
    ).stat().st_size > 0

    for filename in fusion_path_reports
)

FUSION_PATH_CONFIGURATION_COMPLETED = True


# ==================================================
# 20. PRINT FINAL SUMMARY
# ==================================================

print("\n" + "=" * 70)

print("FINAL MULTIMODAL FUSION PATH SUMMARY")

print("=" * 70)

print("\nPROJECT INFORMATION")
print("-" * 70)

print(f"Project root : {PROJECT_ROOT}")

print(f"Visual path  : {VISUAL_FEATURES_DIR}")

print(f"Text path    : {TEXT_FEATURES_DIR}")

print(f"Fusion path  : {FUSION_FEATURES_DIR}")

print(f"Figures path : {FUSION_FIGURES_DIR}")

print(f"Metrics path : {METRICS_DIR}")


print("\nINPUT ARTIFACT INFORMATION")
print("-" * 70)

print(

    f"Visual payload artifacts : "

    f"{len(VISUAL_ARTIFACT_PATHS)}"
)

print(

    f"Text artifacts           : "

    f"{sum(len(p) for p in TEXT_ARTIFACT_PATHS.values()) + len(TEXT_SHARED_ARTIFACT_PATHS)}"
)

print(

    f"Required input files     : "

    f"{len(fusion_input_file_validation_df)}"
)

print(

    f"Available input files    : "

    f"{fusion_input_file_validation_df['passed'].sum()}"
)

print(

    f"Missing or invalid files : "

    f"{(~fusion_input_file_validation_df['passed']).sum()}"
)


print("\nOUTPUT CONFIGURATION")
print("-" * 70)

print(

    f"Output directories       : "

    f"{len(fusion_output_directory_df)}"
)

print(

    f"Writable directories     : "

    f"{fusion_output_directory_df['writable'].sum()}"
)

print(

    f"Planned fusion artifacts : "

    f"{len(FUSION_OUTPUT_ARTIFACT_PATHS)}"
)

print(

    f"Generated reports        : "

    f"{len(fusion_path_reports)}"
)


print("\nVALIDATION RESULTS")
print("-" * 70)

print(

    f"Technical checks : "

    f"{fusion_path_passed_checks}/{fusion_path_total_checks}"
)

print(

    f"Failed checks    : "

    f"{fusion_path_failed_checks}"
)


print("\n[PASS] Project root detected.")

print("[PASS] Visual artifact paths configured.")

print("[PASS] Text artifact paths configured.")

print("[PASS] Metadata paths configured.")

print("[PASS] Required input files validated.")

print("[PASS] Artifact manifests available.")

print("[PASS] Output directories created.")

print("[PASS] Output write access validated.")

print("[PASS] Path configuration reports saved.")

print("\n[INFO] No feature matrices were loaded.")

print("[INFO] No vectorizers were refitted.")

print("[INFO] No input artifacts were modified.")

print("[INFO] Artifact integrity will be verified during loading.")

print("[INFO] Multimodal fusion has not been performed.")

print("\n" + "=" * 70)

print("PATH CONFIGURATION COMPLETED SUCCESSFULLY")

print("=" * 70)

print("\nNext cell: Cell 4 - Load Visual Feature Artifacts")

MULTIMODAL FUSION - PATH CONFIGURATION
[PASS] Library initialization validated.

1. PROJECT ROOT DETECTION
----------------------------------------------------------------------
Working directory : d:\Project\Multimodal_Leaf_Retrieval\notebooks
Project root      : D:\Project\Multimodal_Leaf_Retrieval
[PASS] Project root detected successfully.

2. INPUT DIRECTORY VALIDATION
----------------------------------------------------------------------


,directory,relative_path,exists,passed,status
0,Dataset,dataset,True,True,Available
1,Processed Dataset,dataset/processed,True,True,Available
2,Feature Dataset,dataset/features,True,True,Available
3,Visual Features,dataset/features/visual,True,True,Available
4,Text Features,dataset/features/text,True,True,Available



3. REQUIRED INPUT FILE VALIDATION
----------------------------------------------------------------------


,modality,filename,size_kib,status
0,Dataset,metadata_processed.csv,147.1865,Available
1,Visual,hsv_features.npy,52.1250,Available
2,Visual,glcm_features.npy,58.6250,Available
3,Visual,hu_features.npy,5.8125,Available
4,Visual,hu_raw_moments.npy,5.8125,Available
5,Visual,visual_features_raw.npy,116.3125,Available
6,Visual,visual_features_baseline.npy,116.3125,Available
7,Visual,visual_features_normalized.npy,116.3125,Available
8,Visual,visual_sample_ids.npy,8.6562,Available
9,Visual,visual_feature_metadata.csv,159.1514,Available



4. SUPPORTING ARTIFACTS
----------------------------------------------------------------------


,artifact,filename,relative_path,available,status
0,Visual Manifest JSON,visual_feature_manifest.json,dataset/features/visual/visual_feature_manifest.json,True,Available
1,Visual Manifest CSV,174_visual_feature_artifact_manifest.csv,results/metrics/174_visual_feature_artifact_manifest.csv,True,Available
2,Text Notebook Handoff,347_text_feature_extraction_handoff.json,results/metrics/347_text_feature_extraction_handoff.json,True,Available


... (7 outputs left out)

,validation_check,passed,status
0,Project root detected,True,Passed
1,Required input directories available,True,Passed
2,All required input artifacts available,True,Passed
3,All required input files nonempty,True,Passed
4,Visual artifact manifest available,True,Passed
5,Text artifact manifest available,True,Passed
6,Output directories available,True,Passed
7,Output directories writable,True,Passed
8,Required input paths unique,True,Passed
9,Planned output paths unique,True,Passed



9. PATH CONFIGURATION SUMMARY
----------------------------------------------------------------------


,Indicator,Result
0,Project root detected,True
1,Required input directories,5
2,Available input directories,5
3,Required visual payload artifacts,14
4,Required text artifacts including manifest,17
5,Total required input files,32
6,Available required input files,32
7,Missing or invalid input files,0
8,Visual manifest available,True
9,Text handoff available,True



10. SAVING PATH CONFIGURATION REPORTS
----------------------------------------------------------------------
[SAVED] 348_fusion_input_directory_validation.csv
[SAVED] 349_fusion_input_file_validation.csv
[SAVED] 350_fusion_supporting_artifact_validation.csv
[SAVED] 351_fusion_output_directory_validation.csv
[SAVED] 352_fusion_input_artifact_summary.csv
[SAVED] 353_fusion_planned_artifacts.csv
[SAVED] 354_fusion_path_validation.csv
[SAVED] 355_fusion_path_configuration_summary.csv
[SAVED] 356_fusion_library_versions.csv
[SAVED] 357_fusion_library_validation.csv

FINAL MULTIMODAL FUSION PATH SUMMARY

PROJECT INFORMATION
----------------------------------------------------------------------
Project root : D:\Project\Multimodal_Leaf_Retrieval
Visual path  : D:\Project\Multimodal_Leaf_Retrieval\dataset\features\visual
Text path    : D:\Project\Multimodal_Leaf_Retrieval\dataset\features\text
Fusion path  : D:\Project\Multimodal_Leaf_Retrieval\dataset\features\fusion
Figures path : D:\Projec